# 04 — Engine Inspector：載入 · 檢視 · 測試

載入已建好的 TRT engine，完整印出所有引擎資訊，並用測試圖片驗證推論結果。

**前提**：TensorRT Python bindings（tensorrt 已在 pyproject.toml 中）
```
uv pip install "C:/Users/edisonhsieh/Downloads/TensorRT-10.8.0.43.Windows.win10.cuda-12.8/TensorRT-10.8.0.43/python/tensorrt-10.8.0.43-cp312-none-win_amd64.whl"
```

| Cell | 內容 |
|------|------|
| 1 | 路徑設定與環境準備 |
| 2 | 載入 Engine + 基本資訊 |
| 3 | 詳細張量資訊（所有 IO tensors）|
| 4 | 引擎層列表（Layer Inspector）|
| 5 | Optimization Profile 檢視 |
| 6 | 單張圖推論測試 |
| 7 | 延遲量測（100 warmup + 500 timed runs）|

## 1. 路徑設定與環境準備

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
from src.env import *

TRT_ROOT      = TRTEXEC.parent.parent
ENGINES_DIR.mkdir(exist_ok=True)
TARGET_ENGINE = ENGINE_FP16   # change to ENGINE_FP32 if needed

print(f"Target engine : {TARGET_ENGINE}  exists={TARGET_ENGINE.exists()}")
if TARGET_ENGINE.exists():
    size_mb = TARGET_ENGINE.stat().st_size / 1024**2
    print(f"Engine size   : {size_mb:.2f} MB")

## 2. 載入 Engine + 基本資訊

In [2]:
try:
    import tensorrt as trt
except ImportError:
    raise SystemExit(
        "tensorrt Python package not found.\n"
        "Install with:\n"
        f'  uv pip install "{TRT_ROOT}/python/tensorrt-10.8.0.43-cp312-none-win_amd64.whl"'
    )

TRT_LOGGER = trt.Logger(trt.Logger.VERBOSE)

print(f"TensorRT Python version : {trt.__version__}")
print(f"TRT_LOGGER level        : VERBOSE (change to WARNING to reduce noise)")

with open(TARGET_ENGINE, "rb") as f:
    engine_data = f.read()

runtime = trt.Runtime(TRT_LOGGER)
engine  = runtime.deserialize_cuda_engine(engine_data)

if engine is None:
    raise RuntimeError(f"Failed to deserialize engine: {TARGET_ENGINE}")

print(f"\n{'='*60}")
print(f"Engine loaded successfully")
print(f"  File              : {TARGET_ENGINE}")
print(f"  # IO tensors      : {engine.num_io_tensors}")
print(f"  # Layers          : {engine.num_layers}")
print(f"  # Opt profiles    : {engine.num_optimization_profiles}")
print(f"  Engine name       : {engine.name!r}")
print(f"{'='*60}")

TensorRT Python version : 10.8.0.43
TRT_LOGGER level        : VERBOSE (change to WARNING to reduce noise)

Engine loaded successfully
  File              : engines\H_fp16.engine
  # IO tensors      : 2
  # Layers          : 58
  # Opt profiles    : 1
  Engine name       : 'Unnamed Network 0'


## 3. 詳細張量資訊（所有 IO Tensors）

In [3]:
import numpy as np

DTYPE_MAP = {
    trt.DataType.FLOAT : "FP32",
    trt.DataType.HALF  : "FP16",
    trt.DataType.INT8  : "INT8",
    trt.DataType.INT32 : "INT32",
    trt.DataType.BOOL  : "BOOL",
    trt.DataType.UINT8 : "UINT8",
}
NP_DTYPE_MAP = {
    trt.DataType.FLOAT : np.float32,
    trt.DataType.HALF  : np.float16,
    trt.DataType.INT8  : np.int8,
    trt.DataType.INT32 : np.int32,
    trt.DataType.BOOL  : np.bool_,
    trt.DataType.UINT8 : np.uint8,
}
TORCH_DTYPE_MAP = {
    trt.DataType.FLOAT : "float32",
    trt.DataType.HALF  : "float16",
    trt.DataType.INT8  : "int8",
    trt.DataType.INT32 : "int32",
}

io_tensors = []

print(f"{'Idx':>4}  {'Mode':>6}  {'Name':<30}  {'Shape':<22}  {'Dtype':>5}  {'Bytes':>10}")
print("-" * 85)

for i in range(engine.num_io_tensors):
    name      = engine.get_tensor_name(i)
    shape     = list(engine.get_tensor_shape(name))
    dtype     = engine.get_tensor_dtype(name)
    mode      = engine.get_tensor_mode(name)
    fmt       = engine.get_tensor_format(name)

    np_dtype  = NP_DTYPE_MAP.get(dtype, np.float32)
    elem_size = np.dtype(np_dtype).itemsize
    n_elems   = 1
    for d in shape:
        if d > 0:
            n_elems *= d
    nbytes = n_elems * elem_size

    is_dynamic = any(d < 0 for d in shape)
    shape_str  = str(shape) + ("  [dynamic]" if is_dynamic else "")

    print(f"{i:>4}  {mode.name:>6}  {name:<30}  {shape_str:<22}  {DTYPE_MAP.get(dtype, str(dtype)):>5}  {nbytes:>10,} B")

    io_tensors.append({
        "idx": i, "name": name, "shape": shape,
        "dtype": dtype, "np_dtype": np_dtype, "mode": mode,
        "fmt": fmt, "nbytes": nbytes,
    })

inputs  = [t for t in io_tensors if t["mode"] == trt.TensorIOMode.INPUT]
outputs = [t for t in io_tensors if t["mode"] == trt.TensorIOMode.OUTPUT]

print(f"\nInput tensors  : {len(inputs)}")
print(f"Output tensors : {len(outputs)}")

 Idx    Mode  Name                            Shape                   Dtype       Bytes
-------------------------------------------------------------------------------------
   0   INPUT  images                          [1, 3, 448, 448]         FP32   2,408,448 B
   1  OUTPUT  output                          [1, 6]                   FP32          24 B

Input tensors  : 1
Output tensors : 1


## 4. 引擎層列表（Layer Inspector）

> TensorRT fuses and reorders layers internally; the list reflects the *optimized* execution graph.

In [4]:
import json

inspector = engine.create_engine_inspector()
print(f"Total layers: {engine.num_layers}\n")

def _tensor_str(t):
    """Handle TRT layer JSON tensor entries — may be dict or plain string."""
    if isinstance(t, dict):
        name = t.get("Name") or t.get("name", "?")
        dims = t.get("Dimensions") or t.get("dimensions", "")
        return f"{name} {dims}".strip()
    return str(t)

layer_types = {}
for i in range(engine.num_layers):
    info_str = inspector.get_layer_information(i, trt.LayerInformationFormat.JSON)
    try:
        info   = json.loads(info_str)
        ltype  = info.get("LayerType", "Unknown")
        lname  = info.get("Name", f"layer_{i}")
        layer_types[ltype] = layer_types.get(ltype, 0) + 1

        inputs_s  = ", ".join(_tensor_str(t) for t in info.get("Inputs",  []))
        outputs_s = ", ".join(_tensor_str(t) for t in info.get("Outputs", []))

        print(f"  [{i:3d}] {ltype:<28} | {lname[:45]:<45}")
        if inputs_s:
            print(f"         in : {inputs_s}")
        if outputs_s:
            print(f"         out: {outputs_s}")
    except (json.JSONDecodeError, Exception) as e:
        print(f"  [{i:3d}] (parse error: {e}) {info_str[:80]}")

print(f"\n{'─'*50}")
print("Layer type summary:")
for ltype, count in sorted(layer_types.items(), key=lambda x: -x[1]):
    print(f"  {count:>4}x  {ltype}")

Total layers: 58

  [  0] (parse error: 'str' object has no attribute 'get') "Reformatting CopyNode for Input Tensor 0 to /conv1/Conv + /relu/Relu"

  [  1] (parse error: 'str' object has no attribute 'get') "/conv1/Conv + /relu/Relu"

  [  2] (parse error: 'str' object has no attribute 'get') "/maxpool/MaxPool"

  [  3] (parse error: 'str' object has no attribute 'get') "/layer1/layer1.0/conv1/Conv + /layer1/layer1.0/relu/Relu"

  [  4] (parse error: 'str' object has no attribute 'get') "/layer1/layer1.0/conv2/Conv + /layer1/layer1.0/relu_1/Relu"

  [  5] (parse error: 'str' object has no attribute 'get') "/layer1/layer1.0/conv3/Conv"

  [  6] (parse error: 'str' object has no attribute 'get') "/layer1/layer1.0/downsample/downsample.0/Conv + /layer1/layer1.0/Add + /layer1/
  [  7] (parse error: 'str' object has no attribute 'get') "/layer1/layer1.1/conv1/Conv + /layer1/layer1.1/relu/Relu"

  [  8] (parse error: 'str' object has no attribute 'get') "/layer1/layer1.1/conv2/Conv + /layer

## 5. Optimization Profile 檢視

In [5]:
n_profiles = engine.num_optimization_profiles
print(f"Number of optimization profiles: {n_profiles}")

for p in range(n_profiles):
    print(f"\n  Profile {p}:")
    for t in inputs:
        name = t["name"]
        try:
            shapes = engine.get_tensor_profile_shape(name, p)
            print(f"    {name}")
            print(f"      min : {list(shapes[0])}")
            print(f"      opt : {list(shapes[1])}")
            print(f"      max : {list(shapes[2])}")
        except Exception:
            print(f"    {name}  → static shape {t['shape']} (no profile range)")

Number of optimization profiles: 1

  Profile 0:
    images
      min : [1, 3, 448, 448]
      opt : [1, 3, 448, 448]
      max : [1, 3, 448, 448]


## 6. 單張圖推論測試

取 `TEST_DATASET` 第一張圖，用 TensorRT engine 推論並印出每個類別的機率。  
使用 **PyTorch** 做 CUDA 記憶體管理（不需要 cuda-python）。

In [6]:
import cv2
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available — cannot run TRT inference")

device = torch.device("cuda")
CLASS_NAMES = [str(i) for i in range(1, 10)]

# ── Resolve shapes ───────────────────────────────────────────────
in_tensor  = inputs[0]
out_tensor = outputs[0]

in_shape  = [d if d > 0 else 1 for d in in_tensor["shape"]]
out_shape = [d if d > 0 else 1 for d in out_tensor["shape"]]
_, MODEL_C, MODEL_H, MODEL_W = in_shape
print(f"Input  shape : {in_shape}  dtype={DTYPE_MAP.get(in_tensor['dtype'])}")
print(f"Output shape : {out_shape}  dtype={DTYPE_MAP.get(out_tensor['dtype'])}")

# ── Pick first test image ────────────────────────────────────────
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
image_paths = sorted(p for p in TEST_DATASET.rglob("*") if p.suffix.lower() in IMG_EXTS)
if not image_paths:
    raise FileNotFoundError(f"No images found in {TEST_DATASET}")
img_path = image_paths[0]
print(f"\nTest image   : {img_path.name}")

# ── Preprocess ──────────────────────────────────────────────────
img = cv2.imread(str(img_path))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (MODEL_W, MODEL_H))
img = img.astype(np.float32) / 255.0
inp_np = np.ascontiguousarray(np.transpose(img, (2, 0, 1))[None, ...])

in_torch_dtype  = getattr(torch, TORCH_DTYPE_MAP.get(in_tensor["dtype"],  "float32"))
out_torch_dtype = getattr(torch, TORCH_DTYPE_MAP.get(out_tensor["dtype"], "float32"))

inp_dev = torch.from_numpy(inp_np).to(dtype=in_torch_dtype, device=device)
out_dev = torch.empty(out_shape, dtype=out_torch_dtype, device=device)

# ── TRT execution context ────────────────────────────────────────
context = engine.create_execution_context()
context.set_input_shape(in_tensor["name"], in_shape)
context.set_tensor_address(in_tensor["name"],  inp_dev.data_ptr())
context.set_tensor_address(out_tensor["name"], out_dev.data_ptr())

stream = torch.cuda.Stream(device=device)
with torch.cuda.stream(stream):
    ok = context.execute_async_v3(stream.cuda_stream)
stream.synchronize()

if not ok:
    raise RuntimeError("execute_async_v3 returned False")

# ── Postprocess ─────────────────────────────────────────────────
logits = out_dev.cpu().numpy().reshape(-1).astype(np.float32)
if logits.min() < 0 or not np.isclose(logits.sum(), 1.0, atol=1e-3):
    logits = logits - logits.max()
    probs  = np.exp(logits) / np.exp(logits).sum()
else:
    probs = logits

pred_idx   = int(np.argmax(probs))
pred_label = CLASS_NAMES[pred_idx] if pred_idx < len(CLASS_NAMES) else str(pred_idx)

print(f"\nRaw output   : {out_dev.cpu().numpy().reshape(-1)}")
print(f"\nClass probabilities:")
for i, p in enumerate(probs):
    mark  = " <<<" if i == pred_idx else ""
    label = CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i)
    print(f"  class {label}: {p*100:6.2f}%{mark}")
print(f"\nPrediction   : class={pred_label}  confidence={probs[pred_idx]*100:.2f}%")

Input  shape : [1, 3, 448, 448]  dtype=FP32
Output shape : [1, 6]  dtype=FP32

Test image   : Defect0000001 (1503)_34tr.jpg

Raw output   : [   3.703125   32.15625   158.5      -108.5        85.125    -212.75    ]

Class probabilities:
  class 1:   0.00%
  class 2:   0.00%
  class 3: 100.00% <<<
  class 4:   0.00%
  class 5:   0.00%
  class 6:   0.00%

Prediction   : class=3  confidence=100.00%


d:\tensorrt\.venv\Lib\site-packages\torch\cuda\__init__.py:384: UserWarning: Found GPU0 NVIDIA GeForce RTX 5070 Laptop GPU which is of compute capability (CC) 12.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 5.0 which supports hardware CC >=5.0,<6.0 except {5.3}
- 6.0 which supports hardware CC >=6.0,<7.0 except {6.2}
- 6.1 which supports hardware CC >=6.1,<7.0 except {6.2}
- 7.0 which supports hardware CC >=7.0,<8.0 except {7.2}
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 13.0, 13.2
  _warn_unsupported_code(d, device_cc, code_ccs)
d:\tensorrt\.venv\Lib\site-packages\torch\cuda\__init__.py:502: UserWarning: 
NVIDIA GeForce 

## 7. 延遲量測（100 warmup + 500 timed runs）

使用 **PyTorch CUDA Events** 量測 GPU kernel 時間，另外量測含 H2D + D2H 的端到端延遲。

In [7]:
import time

WARMUP_RUNS = 100
TIMED_RUNS  = 500

# ── Re-allocate buffers for a clean timing run ───────────────────
inp_dev2 = torch.from_numpy(inp_np).to(dtype=in_torch_dtype, device=device)
out_dev2 = torch.empty(out_shape, dtype=out_torch_dtype, device=device)

ctx2 = engine.create_execution_context()
ctx2.set_input_shape(in_tensor["name"], in_shape)
ctx2.set_tensor_address(in_tensor["name"],  inp_dev2.data_ptr())
ctx2.set_tensor_address(out_tensor["name"], out_dev2.data_ptr())

stream2 = torch.cuda.Stream(device=device)
torch.cuda.synchronize()

# ── Warmup ───────────────────────────────────────────────────────
print(f"Warming up ({WARMUP_RUNS} runs) ...", end=" ", flush=True)
with torch.cuda.stream(stream2):
    for _ in range(WARMUP_RUNS):
        ctx2.execute_async_v3(stream2.cuda_stream)
stream2.synchronize()
print("done")

# ── GPU-only timing (CUDA events) ────────────────────────────────
ev_start = torch.cuda.Event(enable_timing=True)
ev_stop  = torch.cuda.Event(enable_timing=True)

print(f"Timing GPU-only ({TIMED_RUNS} runs) ...", end=" ", flush=True)
with torch.cuda.stream(stream2):
    ev_start.record(stream2)
    for _ in range(TIMED_RUNS):
        ctx2.execute_async_v3(stream2.cuda_stream)
    ev_stop.record(stream2)
stream2.synchronize()
print("done")

gpu_ms_total = ev_start.elapsed_time(ev_stop)
mean_gpu_ms  = gpu_ms_total / TIMED_RUNS

# ── End-to-end wall-clock (H2D + infer + D2H) ───────────────────
inp_cpu = torch.from_numpy(inp_np).to(dtype=in_torch_dtype)
lat_e2e = []
for _ in range(100):
    t0 = time.perf_counter()
    with torch.cuda.stream(stream2):
        inp_dev2.copy_(inp_cpu, non_blocking=True)
        ctx2.execute_async_v3(stream2.cuda_stream)
        out_cpu = out_dev2.cpu()
    stream2.synchronize()
    lat_e2e.append((time.perf_counter() - t0) * 1000)

lat_e2e = np.array(lat_e2e)

print(f"\n{'='*55}")
print(f"Engine       : {TARGET_ENGINE.name}")
print(f"Input shape  : {in_shape}")
print(f"\nGPU-only latency  (CUDA events, {TIMED_RUNS} runs):")
print(f"  mean   : {mean_gpu_ms:.3f} ms")
print(f"  FPS    : {1000/mean_gpu_ms:.1f}")
print(f"\nEnd-to-end latency (H2D+infer+D2H, 100 runs):")
print(f"  mean   : {lat_e2e.mean():.3f} ms")
print(f"  p50    : {np.percentile(lat_e2e, 50):.3f} ms")
print(f"  p95    : {np.percentile(lat_e2e, 95):.3f} ms")
print(f"  p99    : {np.percentile(lat_e2e, 99):.3f} ms")
print(f"  min    : {lat_e2e.min():.3f} ms")
print(f"  max    : {lat_e2e.max():.3f} ms")
print(f"{'='*55}")

Warming up (100 runs) ... done
Timing GPU-only (500 runs) ... done

Engine       : H_fp16.engine
Input shape  : [1, 3, 448, 448]

GPU-only latency  (CUDA events, 500 runs):
  mean   : 1.112 ms
  FPS    : 898.9

End-to-end latency (H2D+infer+D2H, 100 runs):
  mean   : 1.337 ms
  p50    : 1.300 ms
  p95    : 1.515 ms
  p99    : 1.714 ms
  min    : 1.225 ms
  max    : 1.718 ms
